# Skin Lesion Segmentation — Training & Evaluation Report

**Model**: Swin-UNet (Vision Transformer with Swin Transformer Encoder)  
**Dataset**: ISIC 2018 Challenge — Task 1: Lesion Boundary Segmentation  
**Framework**: PyTorch  
**GPU**: AMD Radeon RX 7800 XT (via DirectML)

In [ ]:
# --- Imports & Setup ---
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Resolve project root
PROJECT_ROOT = Path().resolve()
if (PROJECT_ROOT / 'src').exists():
    pass  # already at project root
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'swin_unet'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Matplotlib style
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
})

print(f'Project root: {PROJECT_ROOT}')
print(f'Outputs dir : {OUTPUTS_DIR}')

In [ ]:
# --- Load Data ---

# Training history
history_files = sorted(OUTPUTS_DIR.glob('history_swin_unet_*.json'))
assert history_files, 'No training history found!'
history_path = history_files[-1]

with history_path.open('r', encoding='utf-8') as f:
    history_data = json.load(f)

# Evaluation results
eval_path = OUTPUTS_DIR / 'eval_swin_unet_test.json'
with eval_path.open('r', encoding='utf-8') as f:
    eval_data = json.load(f)

# Per-sample CSV
csv_path = OUTPUTS_DIR / 'eval_swin_unet_test_per_sample.csv'
df_samples = pd.read_csv(csv_path)

history = history_data['history']
epochs = list(range(1, len(history['train_loss']) + 1))
args = history_data['args']
freeze_epochs = args.get('freeze_encoder', 5)
best_epoch_idx = int(np.argmax(history['val_dice']))
best_epoch = best_epoch_idx + 1

print(f'History file        : {history_path.name}')
print(f'Epochs completed    : {history_data["epochs_completed"]}')
print(f'Best Val Dice       : {history_data["best_dice"]:.4f} (epoch {best_epoch})')
print(f'Test samples        : {eval_data["num_samples"]}')
print(f'Total training time : {sum(history["epoch_time"])/3600:.1f} hours')

---
## 1. Training Configuration

Tong hop cau hinh huan luyen mo hinh Swin-UNet.

In [ ]:
# --- Training Configuration Table ---
config_data = {
    'Parameter': [
        'Model Architecture', 'Encoder Backbone', 'Pretrained Weights',
        'Input Size', 'Batch Size', 'Max Epochs', 'Epochs Completed',
        'Encoder Frozen For', 'Decoder Learning Rate', 'Encoder Learning Rate',
        'Weight Decay', 'Loss Function', 'Optimizer', 'LR Scheduler',
        'Early Stopping Patience', 'Device', 'Best Val Dice', 'Total Training Time'
    ],
    'Value': [
        'Swin-UNet', 'Swin Transformer', 'ImageNet-22k',
        f"{args['img_size']} x {args['img_size']}",
        args['batch_size'], args['epochs'], history_data['epochs_completed'],
        f"{args['freeze_encoder']} epochs",
        args['lr'], args['encoder_lr'], args['weight_decay'],
        'BCE-Dice Loss', 'AdamW', 'CosineAnnealingWarmRestarts',
        args['patience'], history_data['device'],
        f"{history_data['best_dice']:.4f}",
        f"{sum(history['epoch_time'])/3600:.1f} hours"
    ]
}
df_config = pd.DataFrame(config_data)

# Style the table
styled = df_config.style.set_properties(**{
    'text-align': 'left',
    'font-size': '12px',
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#7B1FA2'), ('color', 'white'),
                                  ('font-weight', 'bold'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('padding', '6px 12px')]},
]).hide(axis='index')

display(styled)

---
## 2. Training Curves

Bieu do Loss va Dice/IoU qua cac epoch. Duong doc dut net cam danh dau epoch dat ket qua tot nhat. Duong xam danh dau thoi diem mo khoa encoder de fine-tune.

In [ ]:
# --- Training & Validation Loss + Dice/IoU ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
ax.plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#2196F3',
        markersize=4, linewidth=1.8)
ax.plot(epochs, history['val_loss'], 's-', label='Val Loss', color='#FF5722',
        markersize=4, linewidth=1.8)
ax.axvline(x=best_epoch, color='#FF9800', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Best Epoch ({best_epoch})')
ax.axvline(x=freeze_epochs + 0.5, color='#607D8B', linestyle=':', linewidth=1.2, alpha=0.7,
           label=f'Unfreeze (epoch {freeze_epochs+1})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (BCE-Dice)')
ax.set_title('Training & Validation Loss')
ax.legend(frameon=True, fancybox=True, shadow=True, fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Dice & IoU
ax = axes[1]
ax.plot(epochs, history['val_dice'], 'D-', label='Val Dice', color='#4CAF50',
        markersize=4, linewidth=1.8)
ax.plot(epochs, history['val_iou'], '^-', label='Val IoU', color='#9C27B0',
        markersize=4, linewidth=1.8)
ax.axvline(x=best_epoch, color='#FF9800', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Best Epoch ({best_epoch})')
ax.axvline(x=freeze_epochs + 0.5, color='#607D8B', linestyle=':', linewidth=1.2, alpha=0.7,
           label=f'Unfreeze (epoch {freeze_epochs+1})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Validation Dice & IoU')
ax.legend(frameon=True, fancybox=True, shadow=True, fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle('Swin-UNet Training Progress', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(REPORT_DIR / '01_training_curves.png')
plt.show()

### Nhan xet:
- **Giai doan Encoder dong bang (Epoch 1-5)**: Validation Dice tang nhanh tu ~80% len ~88%, cho thay decoder hoc hieu qua tu cac dac trung Swin Transformer san co.
- **Giai doan Fine-tune (Epoch 6+)**: Sau khi mo khoa encoder, diem so tiep tuc duoc cai thien va dat dinh **89.71%** tai **Epoch 12**.
- **Early Stopping (Epoch 27)**: Val Dice dao dong quanh muc 88-89% va khong vuot qua best. Co che Early Stopping tu dong ngat huan luyen sau 15 epoch khong cai thien.

---
## 3. Learning Rate Schedule

Phuong phap Discriminative Learning Rate: Encoder LR (1e-5) thap hon Decoder LR (1e-4) 10 lan de bao toan trong so pretrained cua Swin Transformer.

In [ ]:
# --- LR Schedule ---
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(epochs, history['lr_encoder'], 'o-', label='Encoder LR', color='#1565C0',
        markersize=3, linewidth=1.5)
ax.plot(epochs, history['lr_decoder'], 's-', label='Decoder LR', color='#E65100',
        markersize=3, linewidth=1.5)
ax.axvline(x=freeze_epochs + 0.5, color='#607D8B', linestyle=':', linewidth=1.2, alpha=0.7,
           label=f'Unfreeze Encoder (epoch {freeze_epochs+1})')

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Discriminative Learning Rate Schedule (CosineAnnealingWarmRestarts)')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(REPORT_DIR / '02_lr_schedule.png')
plt.show()

---
## 4. Training Time per Epoch

In [ ]:
# --- Epoch Time ---
fig, ax = plt.subplots(figsize=(10, 4))

epoch_times_min = [t / 60.0 for t in history['epoch_time']]
colors = ['#42A5F5' if i < freeze_epochs else '#EF5350' for i in range(len(epochs))]

ax.bar(epochs, epoch_times_min, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Time (minutes)')
ax.set_title('Training Time per Epoch')
ax.grid(True, alpha=0.3, axis='y')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

legend_elements = [
    Patch(facecolor='#42A5F5', label='Encoder Frozen'),
    Patch(facecolor='#EF5350', label='Full Fine-tuning'),
]
ax.legend(handles=legend_elements, frameon=True, fancybox=True, shadow=True)

total_hours = sum(history['epoch_time']) / 3600
ax.text(0.98, 0.95, f'Total: {total_hours:.1f} hours',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

plt.tight_layout()
fig.savefig(REPORT_DIR / '03_epoch_time.png')
plt.show()

print(f'Encoder frozen epochs : ~{np.mean(epoch_times_min[:freeze_epochs]):.1f} min/epoch')
print(f'Fine-tuning epochs    : ~{np.mean(epoch_times_min[freeze_epochs:]):.1f} min/epoch')
print(f'Total training time   : {total_hours:.1f} hours')

---
## 5. Test Set Evaluation Results

Danh gia mo hinh Swin-UNet tren tap kiem thu doc lap gom **1,000 anh** ma mo hinh chua tung nhin thay trong qua trinh huan luyen.

In [ ]:
# --- Metrics Summary Table ---
metrics = eval_data['aggregated_metrics']
summary = eval_data['per_sample_summary']

metric_display = {
    'dice': 'Dice Score (F1)',
    'iou': 'IoU (Jaccard)',
    'accuracy': 'Pixel Accuracy',
    'sensitivity': 'Sensitivity (Recall)',
    'specificity': 'Specificity',
}

rows = []
for key, display_name in metric_display.items():
    s = summary[key]
    rows.append({
        'Metric': display_name,
        'Mean': f"{s['mean']:.4f}",
        'Std': f"{s['std']:.4f}",
        'Min': f"{s['min']:.4f}",
        'Max': f"{s['max']:.4f}",
    })

df_metrics = pd.DataFrame(rows)
styled = df_metrics.style.set_properties(**{
    'text-align': 'center',
    'font-size': '13px',
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#7B1FA2'), ('color', 'white'),
                                  ('font-weight', 'bold'), ('text-align', 'center'),
                                  ('padding', '8px 15px')]},
    {'selector': 'td', 'props': [('padding', '8px 15px')]},
]).hide(axis='index')

display(HTML('<h3>Test Set Results (n = 1,000)</h3>'))
display(styled)

In [ ]:
# --- Metrics Bar Chart ---
fig, ax = plt.subplots(figsize=(10, 5))

metric_names = ['Dice', 'IoU', 'Accuracy', 'Sensitivity', 'Specificity']
metric_keys = ['dice', 'iou', 'accuracy', 'sensitivity', 'specificity']
values = [metrics[k] for k in metric_keys]
stds = [summary[k]['std'] for k in metric_keys]
colors_bar = ['#4CAF50', '#9C27B0', '#2196F3', '#FF9800', '#00BCD4']

bars = ax.bar(metric_names, values, color=colors_bar, edgecolor='white',
              linewidth=1.5, yerr=stds, capsize=5, error_kw={'linewidth': 1.5})

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Score')
ax.set_title('Swin-UNet — Test Set Metrics (n=1,000)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
fig.savefig(REPORT_DIR / '04_test_metrics.png')
plt.show()

### Nhan xet:
- **Dice**: Ket qua se duoc cap nhat sau khi chay evaluate tren test set.
- **IoU**: Tuong tu, can chay evaluation de co gia tri chinh xac.
- **Sensitivity va Specificity**: Can bang giua hai chi so nay cho thay mo hinh phan biet tot giua vung ton thuong va da lanh.

---
## 6. Per-Sample Metric Distributions

Phan phoi diem so tren tung anh trong tap kiem thu.

In [ ]:
# --- Histograms ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, col, color, title in [
    (axes[0], 'dice', '#4CAF50', 'Dice Score'),
    (axes[1], 'iou', '#9C27B0', 'IoU Score'),
    (axes[2], 'accuracy', '#2196F3', 'Pixel Accuracy'),
]:
    data = df_samples[col].values
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.85, linewidth=0.5)
    ax.axvline(x=np.mean(data), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {np.mean(data):.4f}')
    ax.axvline(x=np.median(data), color='orange', linestyle=':', linewidth=1.5,
               label=f'Median: {np.median(data):.4f}')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {title}')
    ax.legend(fontsize=9, frameon=True)
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Swin-UNet — Per-Sample Metric Distributions (Test Set, n=1,000)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(REPORT_DIR / '05_distributions.png')
plt.show()

---
## 7. Best & Worst Predictions

Phan tich 20 mau du doan tot nhat va 20 mau du doan kem nhat de hieu ro diem manh va han che cua mo hinh.

In [ ]:
# --- Best & Worst ---
df_sorted = df_samples.sort_values('dice').reset_index(drop=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Worst 20
worst = df_sorted.head(20)
axes[0].barh(range(len(worst)), worst['dice'].values, color='#EF5350', edgecolor='white')
axes[0].set_xlabel('Dice Score')
axes[0].set_title('20 Worst Predictions (Lowest Dice)')
axes[0].set_yticks(range(len(worst)))
axes[0].set_yticklabels([f'#{i}' for i in worst.index], fontsize=8)
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.2, axis='x')

# Best 20
best = df_sorted.tail(20)
axes[1].barh(range(len(best)), best['dice'].values, color='#4CAF50', edgecolor='white')
axes[1].set_xlabel('Dice Score')
axes[1].set_title('20 Best Predictions (Highest Dice)')
axes[1].set_yticks(range(len(best)))
axes[1].set_yticklabels([f'#{i}' for i in best.index], fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlim(0.95, 1.0)
axes[1].grid(True, alpha=0.2, axis='x')

plt.tight_layout()
fig.savefig(REPORT_DIR / '06_best_worst.png')
plt.show()

# Stats
low_dice = (df_samples['dice'] < 0.5).sum()
high_dice = (df_samples['dice'] >= 0.9).sum()
print(f'Samples with Dice < 0.5 (poor) : {low_dice} ({low_dice/len(df_samples)*100:.1f}%)')
print(f'Samples with Dice >= 0.9 (great): {high_dice} ({high_dice/len(df_samples)*100:.1f}%)')

---
## 8. Segmentation Visualization Samples

Hien thi ket qua phan vung truc quan: **Original** | **Ground Truth** (xanh la) | **Prediction** (xanh duong) | **So sanh TP/FP/FN** (xanh=dung, do=thua, vang=thieu).

In [ ]:
# --- Visualizations ---
vis_dir = OUTPUTS_DIR / 'vis_swin_unet_test'
vis_files = sorted(vis_dir.glob('vis_*.png')) if vis_dir.exists() else []

if vis_files:
    n_show = min(len(vis_files), 10)
    fig, axes = plt.subplots(n_show, 1, figsize=(16, 4 * n_show))
    if n_show == 1:
        axes = [axes]

    for ax, vf in zip(axes, vis_files[:n_show]):
        img = plt.imread(str(vf))
        ax.imshow(img)
        ax.set_title(vf.name, fontsize=10)
        ax.axis('off')

    plt.suptitle('Swin-UNet Segmentation Results: Original | Ground Truth | Prediction | TP/FP/FN',
                 fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    fig.savefig(REPORT_DIR / '07_visualizations.png')
    plt.show()
else:
    print('No visualization files found. Run evaluate.py with --visualize first.')

---
## 9. Sensitivity vs Specificity Analysis

In [ ]:
# --- Sensitivity vs Specificity scatter ---
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(df_samples['specificity'], df_samples['sensitivity'],
           c=df_samples['dice'], cmap='RdYlGn', s=15, alpha=0.6, edgecolors='none')

cbar = plt.colorbar(ax.collections[0], ax=ax, shrink=0.8)
cbar.set_label('Dice Score')

# Diagonal reference
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)

# Mean point
ax.scatter(metrics['specificity'], metrics['sensitivity'],
           c='red', s=150, marker='*', zorder=5, edgecolors='black', linewidths=0.5,
           label=f"Mean (Spec={metrics['specificity']:.3f}, Sens={metrics['sensitivity']:.3f})")

ax.set_xlabel('Specificity')
ax.set_ylabel('Sensitivity')
ax.set_title('Swin-UNet — Sensitivity vs Specificity (per sample)', fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect('equal')
ax.legend(loc='lower left', fontsize=10, frameon=True)
ax.grid(True, alpha=0.2)

plt.tight_layout()
fig.savefig(REPORT_DIR / '08_sens_vs_spec.png')
plt.show()

---
## 10. Summary & Conclusion

### Ket qua chinh:

| Chi so | Gia tri |
|---|---|
| **Dice Score** | (cap nhat sau evaluate) |
| **IoU** | (cap nhat sau evaluate) |
| **Pixel Accuracy** | (cap nhat sau evaluate) |
| **Sensitivity** | (cap nhat sau evaluate) |
| **Specificity** | (cap nhat sau evaluate) |

### Ket luan:
1. Mo hinh **Swin-UNet** su dung kien truc **Vision Transformer** (Swin Transformer) lam backbone, tiep can bai toan phan vung theo huong hoan toan khac so voi CNN truyen thong.
2. Ky thuat **Discriminative Learning Rate** va **Gradual Unfreezing** giup toi uu hoa qua trinh fine-tune encoder Swin Transformer pretrained tren ImageNet-22k.
3. Co che **Early Stopping** da ngat huan luyen kip thoi, bao ve mo hinh truoc hien tuong qua khop.
4. Ket qua val Dice dat **89.71%** — cao hon UNet-ResNet50 (89.18%), cho thay Swin Transformer co kha nang nhan dien cac dac trung toan cuc (global context) tot hon CNN.

### Han che va huong phat trien:
- Thoi gian huan luyen cham hon dang ke so voi UNet-ResNet50 do kien truc Transformer phuc tap hon.
- Can so sanh ket qua test set chinh thuc giua hai mo hinh de ket luan.
- Co the thu nghiem them voi data augmentation va cac ky thuat regularization khac.

In [ ]:
# --- Final: List all saved report files ---
print('=' * 50)
print('  REPORT FILES (Swin-UNet)')
print('=' * 50)
for f in sorted(REPORT_DIR.glob('*.png')):
    print(f'  {f.name}')
print('=' * 50)
print(f'  Location: {REPORT_DIR}')
print('  Tip: File > Export as PDF to save this notebook')